In [ ]:
# NUESTRO agente LLM en la G4 (RTX Pro 6000). Offline, wheels+modelo de datasets públicos.
import json, os, subprocess, sys, time
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1","true"}
NOTEBOOK_START = time.time()
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["MPLBACKEND"] = "Agg"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in ["/usr/local/nvidia/lib64", os.environ.get("LIBRARY_PATH","")] if e)
OFFLINE_SOFT_MIN = float(os.environ.get("TAAF_OFFLINE_SOFT_MIN", "30"))

def find_input(marker_names):
    for dp, dns, fns in os.walk("/kaggle/input"):
        names = set(dns) | set(fns)
        for m in marker_names:
            if m in names:
                return Path(dp)
    return None

COMP_ROOT = find_input(["arc_agi_3_wheels"])
WHEELHOUSE = find_input(["vllm"]) or None
# el snapshot del modelo trae config.json + *.safetensors
MODEL_DIR = None
for dp, dns, fns in os.walk("/kaggle/input"):
    if "config.json" in fns and any(f.endswith(".safetensors") for f in fns):
        MODEL_DIR = Path(dp); break
assert COMP_ROOT, "wheelhouse de competencia no encontrado"
print("COMP_ROOT=", COMP_ROOT, "MODEL_DIR=", MODEL_DIR)

# arc-agi (competencia) offline
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-index",
    "--no-warn-conflicts","--disable-pip-version-check",
    f"--find-links={COMP_ROOT/'arc_agi_3_wheels'}","arc-agi"], stdout=subprocess.DEVNULL)
import arc_agi
print("arc_agi OK")


In [ ]:
import os
os.makedirs('/kaggle/working/src/arc3', exist_ok=True)
SOURCES = {
 "src/arc3/__init__.py": "\"\"\"arc3: utilidades para ARC-AGI-3 (Kaggle arc-prize-2026-arc-agi-3).\n\nM\u00f3dulos:\n  env      -> descubrimiento y ejecuci\u00f3n local de environments (arcengine/arc_agi)\n  features -> feature engineering sobre frames 64x64 y transiciones (s, a, s')\n  probe    -> pol\u00edtica de sondeo que genera el dataset de features por juego\n\"\"\"\n\nfrom .features import (\n    connected_components,\n    grid_features,\n    frame_to_grid,\n    transition_features,\n)\n\n__all__ = [\n    \"connected_components\",\n    \"grid_features\",\n    \"frame_to_grid\",\n    \"transition_features\",\n]\n",
 "src/arc3/agent.py": "\"\"\"GraphExplorer: agente de exploraci\u00f3n de grafo de estados para ARC-AGI-3.\n\nS\u00edntesis de lo mejor del leaderboard p\u00fablico (ver docs/STRATEGY.md):\n  - Grafo de estados con hashing enmascarado (borde 3px + m\u00e1scara de contador aprendida),\n    BFS sobre el grafo aprendido para volver a nodos con acciones pendientes, y replay\n    tras RESET aprovechando el determinismo de los juegos.  [estilo v47, LB 0.54]\n  - Clicks por componentes conexas ordenadas por button-likeness (compacto+peque\u00f1o+color\n    raro) + rejilla gruesa de cobertura; supresi\u00f3n \"deadsig\" de clases estructuralmente\n    inertes con protecci\u00f3n de clases alguna vez efectivas.   [estilo 2\u00ba milestone]\n  - Orden de acciones simples por P(cambio) aprendida online; no-ops se hunden, no se podan.\n\nL\u00f3gica pura sobre numpy: el runner (local o gateway) le pasa frames y ejecuta lo que elige.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom typing import Any, Optional\n\nimport numpy as np\n\nfrom .features import connected_components\n\nGRID = 64\nBORDER = 3           # borde enmascarado del hash: HUD/contadores viven ah\u00ed\nCOUNTER_WARMUP = 12  # transiciones para aprender la m\u00e1scara de contador\nCOUNTER_FRACTION = 0.8   # celda contador si cambia en >=80% de las transiciones\nCOUNTER_MAX_INTERIOR = 0.2  # la m\u00e1scara aprendida no puede tapar >20% del interior\nCLICK_CAP = 64       # candidatos de click por nodo\nDEAD_K = 2           # clase de click muerta tras K usos inertes\nMAX_EXHAUSTED_RESETS = 40   # reinicios diversificados antes de rendirse\nRESET_LOOP_BREAK = 30\n\n# ids de acci\u00f3n: 0=RESET, 1..5 y 7 simples, 6=click(x,y)\nSIMPLE_IDS = (1, 2, 3, 4, 5, 7)\nRESET_KEY = (0, -1, -1)\n\n\nclass _Node:\n    __slots__ = (\"pending\", \"tried\")\n\n    def __init__(self) -> None:\n        self.pending: deque[tuple[int, int, int]] = deque()\n        self.tried: set[tuple[int, int, int]] = set()\n\n\nclass GraphExplorer:\n    \"\"\"Elige (action_id, x, y). El caller ejecuta y devuelve el frame en el pr\u00f3ximo choose().\"\"\"\n\n    def __init__(self, game_id: str = \"\", max_actions: int = 15000) -> None:\n        self.game_id = game_id\n        self.max_actions = max_actions\n        self.actions_taken = 0\n\n        self._nodes: dict[int, _Node] = {}\n        self._edges: dict[tuple[int, tuple[int, int, int]], int] = {}\n        self._adj: dict[int, list[tuple[tuple[int, int, int], int]]] = {}\n\n        self._counter_counts = np.zeros((GRID, GRID), dtype=np.int32)\n        self._counter_seen = 0\n        self._counter_mask: Optional[np.ndarray] = None\n\n        # stats por acci\u00f3n simple: [cambios, usos, nodos_nuevos]\n        self._act_stats: dict[int, list[int]] = {a: [0, 0, 0] for a in SIMPLE_IDS}\n        # deadsig por clase estructural de click (color, size, is_rect)\n        self._dead_sigs: dict[tuple[int, int, bool], int] = {}\n        self._eff_sigs: set[tuple[int, int, bool]] = set()\n\n        self._last_key: Optional[int] = None\n        self._last_action: Optional[tuple[int, int, int]] = None\n        self._last_grid: Optional[np.ndarray] = None\n        self._last_levels = 0\n        self._replay: deque[tuple[int, int, int]] = deque()\n        self._replay_target: Optional[int] = None\n        self._exhausted_resets = 0\n        self._consecutive_resets = 0\n        self._salt = 0   # perturba el orden de candidatos en cada reinicio diversificado\n        self.done = False\n\n    # ---------- hashing ----------\n\n    def _mask(self) -> np.ndarray:\n        m = np.zeros((GRID, GRID), dtype=bool)\n        m[:BORDER, :] = m[-BORDER:, :] = m[:, :BORDER] = m[:, -BORDER:] = True\n        if self._counter_mask is not None:\n            m |= self._counter_mask\n        return m\n\n    def _key(self, grid: np.ndarray) -> int:\n        g = grid.copy()\n        g[self._mask()] = 0\n        return hash(g.tobytes())\n\n    def _learn_counter_mask(self, prev: np.ndarray, nxt: np.ndarray) -> None:\n        if self._counter_mask is not None or prev is None:\n            return\n        self._counter_counts += prev != nxt\n        self._counter_seen += 1\n        if self._counter_seen >= COUNTER_WARMUP:\n            cand = self._counter_counts >= COUNTER_FRACTION * self._counter_seen\n            cand[:BORDER, :] = cand[-BORDER:, :] = cand[:, :BORDER] = cand[:, -BORDER:] = False\n            interior = (GRID - 2 * BORDER) ** 2\n            # si \"todo cambia siempre\" (animaci\u00f3n global) la m\u00e1scara ser\u00eda in\u00fatil: borde solo\n            self._counter_mask = cand if cand.sum() <= COUNTER_MAX_INTERIOR * interior \\\n                else np.zeros((GRID, GRID), dtype=bool)\n\n    # ---------- candidatos ----------\n\n    def _click_sig(self, obj: dict[str, Any]) -> tuple[int, int, bool]:\n        y0, x0, y1, x1 = obj[\"bbox\"]\n        is_rect = obj[\"size\"] == (y1 - y0 + 1) * (x1 - x0 + 1)\n        return (obj[\"color\"], obj[\"size\"], is_rect)\n\n    def _click_candidates(self, grid: np.ndarray) -> list[tuple[int, int, int]]:\n        counts = np.bincount(grid.ravel(), minlength=16)\n        background = int(counts.argmax())\n        total = grid.size\n        objs = connected_components(grid, background)\n        scored = []\n        for o in objs:\n            sig = self._click_sig(o)\n            if self._dead_sigs.get(sig, 0) >= DEAD_K and sig not in self._eff_sigs:\n                continue\n            rarity = 1.0 - counts[o[\"color\"]] / total\n            y0, x0, y1, x1 = o[\"bbox\"]\n            fill = o[\"size\"] / ((y1 - y0 + 1) * (x1 - x0 + 1))\n            size_score = 1.0 if o[\"size\"] <= 4 else 0.8 if o[\"size\"] <= 16 else \\\n                0.5 if o[\"size\"] <= 64 else 0.25 if o[\"size\"] <= 256 else 0.0\n            score = 0.4 * rarity + 0.3 * size_score + 0.3 * fill\n            cy, cx = o[\"centroid\"]\n            scored.append((score, int(round(cx)), int(round(cy))))\n        scored.sort(key=lambda t: -t[0])\n        cands = [(6, x, y) for _, x, y in scored[:CLICK_CAP]]\n        # rejilla de cobertura; se densifica con cada reinicio diversificado (salt)\n        stride = 8 if self._salt == 0 else 4 if self._salt < 3 else 2\n        offset = (self._salt * 3) % max(stride, 1)\n        cap = CLICK_CAP if self._salt == 0 else CLICK_CAP * 4\n        for gy in range(offset, GRID, stride):\n            for gx in range(offset, GRID, stride):\n                if len(cands) >= cap:\n                    break\n                c = (6, gx, gy)\n                if c not in cands:\n                    cands.append(c)\n        return cands[:cap]\n\n    def _simple_order(self, available: list[int]) -> list[int]:\n        acts = [a for a in SIMPLE_IDS if not available or a in available]\n        # rotaci\u00f3n por salt: cada reinicio diversificado prueba un orden base distinto,\n        # as\u00ed el desempate entre acciones no probadas genera trayectorias nuevas.\n        if self._salt and acts:\n            r = self._salt % len(acts)\n            acts = acts[r:] + acts[:r]\n\n        def score(a: int) -> float:\n            chg, uses, _new = self._act_stats[a]\n            return 0.5 if uses == 0 else chg / uses\n\n        return sorted(acts, key=lambda a: -score(a))\n\n    def _fill_pending(self, node: _Node, grid: np.ndarray, available: list[int]) -> None:\n        for a in self._simple_order(available):\n            k = (a, -1, -1)\n            if k not in node.tried:\n                node.pending.append(k)\n        if not available or 6 in available:\n            for k in self._click_candidates(grid):\n                if k not in node.tried:\n                    node.pending.append(k)\n\n    # ---------- grafo ----------\n\n    def _record_edge(self, src: int, action: tuple[int, int, int], dst: int) -> None:\n        if (src, action) not in self._edges:\n            self._edges[(src, action)] = dst\n            self._adj.setdefault(src, []).append((action, dst))\n\n    def _bfs_to_pending(self, start: int) -> Optional[list[tuple[int, int, int]]]:\n        \"\"\"Camino m\u00e1s corto (en aristas conocidas) hasta un nodo con pendientes.\"\"\"\n        seen = {start}\n        q: deque[tuple[int, list[tuple[int, int, int]]]] = deque([(start, [])])\n        while q:\n            key, path = q.popleft()\n            node = self._nodes.get(key)\n            if node and node.pending and key != start:\n                return path\n            if len(path) >= 60:\n                continue\n            for action, dst in self._adj.get(key, []):\n                if dst not in seen:\n                    seen.add(dst)\n                    q.append((dst, path + [action]))\n        return None\n\n    # ---------- API ----------\n\n    def choose(\n        self,\n        grid: np.ndarray,\n        state: str,\n        levels_completed: int,\n        available_actions: list[int],\n    ) -> tuple[int, int, int]:\n        \"\"\"Devuelve (action_id, x, y); x=y=-1 para acciones simples/RESET.\"\"\"\n        self.actions_taken += 1\n        if self.actions_taken > self.max_actions:\n            self.done = True\n\n        # --- digerir el resultado de la acci\u00f3n anterior ---\n        if self._last_grid is not None and self._last_action is not None:\n            changed = bool((self._last_grid != grid).any())\n            self._learn_counter_mask(self._last_grid, grid)\n            aid, ax, ay = self._last_action\n            if aid in self._act_stats:\n                self._act_stats[aid][1] += 1\n                self._act_stats[aid][0] += int(changed)\n                self._act_stats[aid][2] += int(self._key(grid) not in self._nodes)\n            if aid == 6:\n                sig = self._sig_at(self._last_grid, ax, ay)\n                if sig is not None:\n                    if changed or levels_completed != self._last_levels:\n                        self._eff_sigs.add(sig)\n                    else:\n                        self._dead_sigs[sig] = self._dead_sigs.get(sig, 0) + 1\n\n        if levels_completed > self._last_levels:\n            # nivel nuevo: lo inerte de antes puede ser la clave ahora\n            self._dead_sigs.clear()\n            self._eff_sigs.clear()\n            self._replay.clear()\n            self._replay_target = None\n        self._last_levels = levels_completed\n\n        key = self._key(grid)\n        if self._last_key is not None and self._last_action is not None:\n            self._record_edge(self._last_key, self._last_action, key)\n\n        # --- game over / not played ---\n        if state in (\"NOT_PLAYED\", \"GAME_OVER\"):\n            self._consecutive_resets += 1\n            if self._consecutive_resets >= RESET_LOOP_BREAK:\n                self.done = True\n            return self._commit(grid, key, RESET_KEY)\n        self._consecutive_resets = 0\n\n        # --- replay en curso (verificando determinismo) ---\n        if self._replay:\n            if self._replay_target is not None and key != self._replay_target:\n                self._replay.clear()  # el mundo no sigui\u00f3 el grafo: abortar replay\n                self._replay_target = None\n            else:\n                action = self._replay.popleft()\n                self._replay_target = self._edges.get((key, action))\n                return self._commit(grid, key, action)\n\n        node = self._nodes.get(key)\n        if node is None:\n            node = _Node()\n            self._nodes[key] = node\n            self._fill_pending(node, grid, available_actions)\n\n        if node.pending:\n            action = node.pending.popleft()\n            node.tried.add(action)\n            return self._commit(grid, key, action)\n\n        # nodo agotado: BFS al nodo pendiente m\u00e1s cercano\n        path = self._bfs_to_pending(key)\n        if path:\n            self._replay = deque(path)\n            action = self._replay.popleft()\n            self._replay_target = self._edges.get((key, action))\n            return self._commit(grid, key, action)\n\n        # grafo alcanzable agotado: reinicio DIVERSIFICADO. En juegos deterministas,\n        # re-explorar con el mismo orden repetir\u00eda la trayectoria; subimos el salt para\n        # densificar clicks y perturbar el orden, y abrimos de nuevo la exploraci\u00f3n\n        # (olvidamos pending/tried; conservamos dead/eff sigs y la m\u00e1scara aprendida).\n        self._exhausted_resets += 1\n        self._salt += 1\n        self._nodes.clear()\n        self._edges.clear()\n        self._adj.clear()\n        self._replay.clear()\n        self._replay_target = None\n        if self._exhausted_resets >= MAX_EXHAUSTED_RESETS:\n            self.done = True\n        return self._commit(grid, key, RESET_KEY)\n\n    def _sig_at(self, grid: np.ndarray, x: int, y: int) -> Optional[tuple[int, int, bool]]:\n        counts = np.bincount(grid.ravel(), minlength=16)\n        background = int(counts.argmax())\n        if not (0 <= x < GRID and 0 <= y < GRID) or grid[y, x] == background:\n            return None\n        for o in connected_components(grid, background):\n            y0, x0, y1, x1 = o[\"bbox\"]\n            if y0 <= y <= y1 and x0 <= x <= x1:\n                return self._click_sig(o)\n        return None\n\n    def _commit(self, grid: np.ndarray, key: int, action: tuple[int, int, int]) -> tuple[int, int, int]:\n        self._last_grid = grid.copy()\n        self._last_key = key\n        self._last_action = action\n        return action\n",
 "src/arc3/env.py": "\"\"\"Descubrimiento y ejecuci\u00f3n local de environments ARC-AGI-3.\n\nEnvuelve arc_agi.LocalEnvironmentWrapper para jugar los juegos de\nenvironment_files/ sin API remota (igual que har\u00e1 el rerun de Kaggle offline).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nfrom arc_agi.local_wrapper import LocalEnvironmentWrapper\nfrom arc_agi.models import EnvironmentInfo\nfrom arcengine import FrameDataRaw, GameAction\n\nlogger = logging.getLogger(\"arc3\")\n\n\ndef discover_environments(env_root: Path) -> list[EnvironmentInfo]:\n    \"\"\"Lista los environments locales a partir de environment_files/<game>/<hash>/metadata.json.\"\"\"\n    infos: list[EnvironmentInfo] = []\n    for meta_path in sorted(env_root.glob(\"*/*/metadata.json\")):\n        meta = json.loads(meta_path.read_text(encoding=\"utf-8\"))\n        # local_dir del metadata es relativo al root del dataset; usamos el real.\n        meta[\"local_dir\"] = str(meta_path.parent)\n        infos.append(EnvironmentInfo.model_validate(meta))\n    return infos\n\n\nclass LocalGame:\n    \"\"\"Sesi\u00f3n de un juego local: reset/step con FrameDataRaw.\"\"\"\n\n    def __init__(self, info: EnvironmentInfo, seed: int = 0) -> None:\n        self.info = info\n        self.env = LocalEnvironmentWrapper(\n            environment_info=info,\n            logger=logger,\n            scorecard_id=\"local-probe\",\n            seed=seed,\n            save_recording=False,\n        )\n\n    def reset(self) -> Optional[FrameDataRaw]:\n        return self.env.reset()\n\n    def step(\n        self, action: GameAction, x: Optional[int] = None, y: Optional[int] = None\n    ) -> Optional[FrameDataRaw]:\n        data: dict[str, Any] = {\"game_id\": self.info.game_id}\n        if action.is_complex():\n            data[\"x\"] = int(x or 0)\n            data[\"y\"] = int(y or 0)\n        return self.env.step(action, data=data)\n",
 "src/arc3/features.py": "\"\"\"Feature engineering para frames de ARC-AGI-3.\n\nLos frames son grids 64x64 con colores 0..15. Aqu\u00ed se computan:\n  - features por frame (histograma de color, objetos, simetr\u00edas, bordes, entrop\u00eda)\n  - features de transici\u00f3n (s, a, s'): p\u00edxeles cambiados, bbox del cambio,\n    deltas por color y detecci\u00f3n de traslaci\u00f3n (vector de movimiento)\n\nTodo en numpy puro (sin scipy) para poder correr offline en Kaggle sin deps extra.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom typing import Any, Optional, Sequence\n\nimport numpy as np\n\nN_COLORS = 16\nGRID = 64\n# Desplazamientos m\u00e1ximos a testear al detectar traslaci\u00f3n de objetos entre frames.\nMAX_SHIFT = 8\n\n\ndef frame_to_grid(frame: Any) -> np.ndarray:\n    \"\"\"Convierte FrameData.frame (lista de grids; puede traer varios por animaci\u00f3n)\n    al \u00faltimo grid como np.ndarray (64, 64) int8.\"\"\"\n    if frame is None or len(frame) == 0:\n        return np.zeros((GRID, GRID), dtype=np.int8)\n    last = frame[-1]\n    return np.asarray(last, dtype=np.int8)\n\n\ndef connected_components(\n    grid: np.ndarray, background: Optional[int] = None\n) -> list[dict[str, Any]]:\n    \"\"\"Componentes conexas 4-conectadas de celdas del mismo color (ignora el fondo).\n\n    Devuelve una lista de objetos: color, size, bbox (y0, x0, y1, x1), centroid.\n    BFS puro en python: el grid es 64x64, es barato.\n    \"\"\"\n    h, w = grid.shape\n    if background is None:\n        background = int(np.bincount(grid.ravel(), minlength=N_COLORS).argmax())\n    seen = np.zeros((h, w), dtype=bool)\n    objects: list[dict[str, Any]] = []\n    for y in range(h):\n        for x in range(w):\n            if seen[y, x] or grid[y, x] == background:\n                continue\n            color = int(grid[y, x])\n            q = deque([(y, x)])\n            seen[y, x] = True\n            cells = []\n            while q:\n                cy, cx = q.popleft()\n                cells.append((cy, cx))\n                for ny, nx in ((cy - 1, cx), (cy + 1, cx), (cy, cx - 1), (cy, cx + 1)):\n                    if 0 <= ny < h and 0 <= nx < w and not seen[ny, nx] and grid[ny, nx] == color:\n                        seen[ny, nx] = True\n                        q.append((ny, nx))\n            ys = [c[0] for c in cells]\n            xs = [c[1] for c in cells]\n            objects.append(\n                {\n                    \"color\": color,\n                    \"size\": len(cells),\n                    \"bbox\": (min(ys), min(xs), max(ys), max(xs)),\n                    \"centroid\": (float(np.mean(ys)), float(np.mean(xs))),\n                }\n            )\n    objects.sort(key=lambda o: -o[\"size\"])\n    return objects\n\n\ndef _edge_density(grid: np.ndarray) -> float:\n    \"\"\"Fracci\u00f3n de pares vecinos (4-conn) con colores distintos: mide 'estructura'.\"\"\"\n    dh = grid[:, 1:] != grid[:, :-1]\n    dv = grid[1:, :] != grid[:-1, :]\n    return float((dh.sum() + dv.sum()) / (dh.size + dv.size))\n\n\ndef _entropy(counts: np.ndarray) -> float:\n    p = counts[counts > 0].astype(np.float64)\n    p /= p.sum()\n    return float(-(p * np.log2(p)).sum())\n\n\ndef grid_features(grid: np.ndarray, max_objects: int = 8) -> dict[str, Any]:\n    \"\"\"Features escalares de un grid 64x64.\"\"\"\n    counts = np.bincount(grid.ravel(), minlength=N_COLORS)[:N_COLORS]\n    background = int(counts.argmax())\n    objects = connected_components(grid, background)\n    feats: dict[str, Any] = {\n        \"background\": background,\n        \"n_colors\": int((counts > 0).sum()),\n        \"color_entropy\": _entropy(counts),\n        \"edge_density\": _edge_density(grid),\n        \"sym_h\": float((grid == grid[:, ::-1]).mean()),  # simetr\u00eda izquierda-derecha\n        \"sym_v\": float((grid == grid[::-1, :]).mean()),  # simetr\u00eda arriba-abajo\n        \"n_objects\": len(objects),\n    }\n    for c in range(N_COLORS):\n        feats[f\"color_{c}\"] = int(counts[c])\n    for i in range(max_objects):\n        if i < len(objects):\n            o = objects[i]\n            y0, x0, y1, x1 = o[\"bbox\"]\n            feats[f\"obj{i}_color\"] = o[\"color\"]\n            feats[f\"obj{i}_size\"] = o[\"size\"]\n            feats[f\"obj{i}_cy\"], feats[f\"obj{i}_cx\"] = o[\"centroid\"]\n            feats[f\"obj{i}_h\"], feats[f\"obj{i}_w\"] = y1 - y0 + 1, x1 - x0 + 1\n        else:\n            feats[f\"obj{i}_color\"] = -1\n            feats[f\"obj{i}_size\"] = 0\n            feats[f\"obj{i}_cy\"] = feats[f\"obj{i}_cx\"] = -1.0\n            feats[f\"obj{i}_h\"] = feats[f\"obj{i}_w\"] = 0\n    return feats\n\n\ndef _detect_translation(prev: np.ndarray, nxt: np.ndarray, diff: np.ndarray) -> tuple[int, int, float]:\n    \"\"\"Busca el shift (dy, dx) que mejor explica el cambio como traslaci\u00f3n.\n\n    Solo mira la regi\u00f3n cambiada: si nxt == shift(prev) sobre esa regi\u00f3n, hay\n    movimiento de un objeto. Devuelve (dy, dx, score) con score en [0, 1].\n    \"\"\"\n    ys, xs = np.nonzero(diff)\n    if len(ys) == 0:\n        return 0, 0, 0.0\n    best = (0, 0, 0.0)\n    for dy in range(-MAX_SHIFT, MAX_SHIFT + 1):\n        for dx in range(-MAX_SHIFT, MAX_SHIFT + 1):\n            if dy == 0 and dx == 0:\n                continue\n            sy, sx = ys - dy, xs - dx\n            ok = (sy >= 0) & (sy < GRID) & (sx >= 0) & (sx < GRID)\n            if not ok.any():\n                continue\n            match = float((nxt[ys[ok], xs[ok]] == prev[sy[ok], sx[ok]]).mean())\n            if match > best[2]:\n                best = (dy, dx, match)\n    return best\n\n\ndef transition_features(prev_grid: np.ndarray, next_grid: np.ndarray) -> dict[str, Any]:\n    \"\"\"Features del cambio entre dos frames consecutivos.\"\"\"\n    diff = prev_grid != next_grid\n    n_changed = int(diff.sum())\n    feats: dict[str, Any] = {\"n_changed\": n_changed}\n    if n_changed == 0:\n        feats.update(\n            {\"chg_y0\": -1, \"chg_x0\": -1, \"chg_h\": 0, \"chg_w\": 0,\n             \"chg_area_frac\": 0.0, \"move_dy\": 0, \"move_dx\": 0, \"move_score\": 0.0,\n             \"colors_gained\": 0, \"colors_lost\": 0}\n        )\n        return feats\n    ys, xs = np.nonzero(diff)\n    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()\n    feats[\"chg_y0\"], feats[\"chg_x0\"] = int(y0), int(x0)\n    feats[\"chg_h\"], feats[\"chg_w\"] = int(y1 - y0 + 1), int(x1 - x0 + 1)\n    feats[\"chg_area_frac\"] = float(n_changed / diff.size)\n    prev_counts = np.bincount(prev_grid.ravel(), minlength=N_COLORS)[:N_COLORS]\n    next_counts = np.bincount(next_grid.ravel(), minlength=N_COLORS)[:N_COLORS]\n    delta = next_counts.astype(int) - prev_counts.astype(int)\n    feats[\"colors_gained\"] = int((delta > 0).sum())\n    feats[\"colors_lost\"] = int((delta < 0).sum())\n    dy, dx, score = _detect_translation(prev_grid, next_grid, diff)\n    feats[\"move_dy\"], feats[\"move_dx\"], feats[\"move_score\"] = dy, dx, score\n    return feats\n\n\ndef action_effect_summary(rows: Sequence[dict[str, Any]]) -> list[dict[str, Any]]:\n    \"\"\"Resumen por acci\u00f3n a partir de filas de transici\u00f3n: \u00bfqu\u00e9 acciones 'hacen algo'?\n\n    Cada fila debe traer: action_id, n_changed, level_up (bool), game_over (bool).\n    \"\"\"\n    out: list[dict[str, Any]] = []\n    by_action: dict[int, list[dict[str, Any]]] = {}\n    for r in rows:\n        by_action.setdefault(int(r[\"action_id\"]), []).append(r)\n    for action_id, rs in sorted(by_action.items()):\n        n = len(rs)\n        out.append(\n            {\n                \"action_id\": action_id,\n                \"n_uses\": n,\n                \"p_change\": float(np.mean([r[\"n_changed\"] > 0 for r in rs])),\n                \"avg_pixels_changed\": float(np.mean([r[\"n_changed\"] for r in rs])),\n                \"p_level_up\": float(np.mean([bool(r.get(\"level_up\")) for r in rs])),\n                \"p_game_over\": float(np.mean([bool(r.get(\"game_over\")) for r in rs])),\n            }\n        )\n    return out\n",
 "src/arc3/llm_agent.py": "\"\"\"LLMAgent: agente VLM con features objetuales inyectadas + fallback a GraphExplorer.\n\nDise\u00f1o (s\u00edntesis de lo que punt\u00faa alto + nuestro diferenciador):\n  - Pol\u00edtica = VLM (OpenAI-compatible, servido con vLLM). Cada turno ve la imagen del\n    frame Y la descripci\u00f3n textual de objetos/transici\u00f3n (arc3.llm_prompt).\n  - Cola de plan: ejecuta 1-3 acciones planeadas sin re-llamar al LLM (ahorra inferencias);\n    se aborta si el frame no cambi\u00f3 o el estado se repite.\n  - Memoria failed-state: (frame_hash, acci\u00f3n) inefectiva se recuerda y se le comunica al\n    LLM; evita repetir lo que no funciona en ese estado exacto.\n  - Fallback: ante CUALQUIER fallo del LLM (excepci\u00f3n, timeout, JSON vac\u00edo, plan in\u00fatil)\n    delega en GraphExplorer \u2014 nunca crashea ni se queda sin acci\u00f3n.\n\n`chat_fn(system, user_text, image_data_uri) -> str` se inyecta (el notebook la implementa\ncon el cliente vLLM); as\u00ed este m\u00f3dulo es testeable sin GPU.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nfrom collections import deque\nfrom typing import Any, Callable, Optional\n\nimport numpy as np\n\nfrom .agent import GraphExplorer\nfrom .features import frame_to_grid\nfrom .llm_prompt import (\n    ACTION_NAMES,\n    SYSTEM_PROMPT,\n    build_user_text,\n    frame_png_data_uri,\n    parse_actions,\n)\n\nChatFn = Callable[[str, str, Optional[str]], str]\n\n\ndef _hash(grid: np.ndarray, border: int = 3) -> str:\n    g = grid[border:-border, border:-border] if border else grid\n    return hashlib.sha1(g.tobytes()).hexdigest()\n\n\nclass LLMAgent:\n    def __init__(\n        self,\n        game_id: str,\n        chat_fn: ChatFn,\n        max_actions: int = 15000,\n        plan_max: int = 3,\n        use_image: bool = True,\n    ) -> None:\n        self.game_id = game_id\n        self.chat_fn = chat_fn\n        self.max_actions = max_actions\n        self.plan_max = plan_max\n        self.use_image = use_image\n        self.actions_taken = 0\n        self.done = False\n\n        self._fallback = GraphExplorer(game_id, max_actions=max_actions)\n        self._plan: deque[dict[str, Any]] = deque()\n        self._failed: dict[str, set[str]] = {}   # frame_hash -> {failure_key}\n        self._prev_grid: Optional[np.ndarray] = None\n        self._prev_hash: Optional[str] = None\n        self._prev_action: Optional[dict[str, Any]] = None\n        self._prev_levels = 0\n        self._llm_calls = 0\n        self._llm_fails = 0\n\n    # ----- memoria de inefectividad -----\n\n    def _fail_key(self, action: dict[str, Any]) -> str:\n        if action[\"id\"] == 6:\n            return f\"click@{action.get('x')},{action.get('y')}\"\n        return f\"a{action['id']}\"\n\n    def _ineffective(self, frame_hash: str) -> list[str]:\n        return sorted(self._failed.get(frame_hash, set()))\n\n    def _digest_previous(self, grid: np.ndarray, levels: int) -> None:\n        if self._prev_grid is None or self._prev_action is None:\n            return\n        changed = bool((self._prev_grid != grid).any())\n        leveled = levels != self._prev_levels\n        if not changed and not leveled and self._prev_hash is not None:\n            self._failed.setdefault(self._prev_hash, set()).add(\n                self._fail_key(self._prev_action))\n        if not changed:  # plan que no mueve nada se aborta\n            self._plan.clear()\n\n    # ----- API principal -----\n\n    def choose(\n        self,\n        grid: np.ndarray,\n        state: str,\n        levels_completed: int,\n        available_actions: list[int],\n    ) -> tuple[int, int, int]:\n        self.actions_taken += 1\n        if self.actions_taken > self.max_actions:\n            self.done = True\n\n        self._digest_previous(grid, levels_completed)\n        if levels_completed > self._prev_levels:\n            self._failed.clear()   # lo inefectivo en un nivel puede servir en el siguiente\n            self._plan.clear()\n        self._prev_levels = levels_completed\n\n        if state in (\"NOT_PLAYED\", \"GAME_OVER\"):\n            return self._emit(grid, {\"id\": 0})\n\n        frame_hash = _hash(grid)\n\n        # 1) plan pendiente que siga siendo legal y no inefectivo\n        while self._plan:\n            a = self._plan.popleft()\n            if available_actions and a[\"id\"] not in available_actions and a[\"id\"] != 6:\n                continue\n            if self._fail_key(a) in self._failed.get(frame_hash, set()):\n                continue\n            return self._emit(grid, a)\n\n        # 2) consultar al LLM\n        action = self._ask_llm(grid, frame_hash, available_actions, levels_completed)\n        if action is not None:\n            return self._emit(grid, action)\n\n        # 3) fallback: GraphExplorer (nunca sin acci\u00f3n)\n        self._llm_fails += 1\n        aid, x, y = self._fallback.choose(grid, state, levels_completed, available_actions)\n        self.done = self.done or self._fallback.done\n        act = {\"id\": aid} if aid != 6 else {\"id\": 6, \"x\": x, \"y\": y}\n        # no re-emitir por fallback (ya viene del fallback); solo registrar estado\n        self._prev_grid = grid.copy()\n        self._prev_hash = frame_hash\n        self._prev_action = act\n        return aid, x, y\n\n    def _ask_llm(\n        self, grid: np.ndarray, frame_hash: str,\n        available_actions: list[int], levels: int,\n    ) -> Optional[dict[str, Any]]:\n        try:\n            user = build_user_text(grid, self._prev_grid, available_actions, levels,\n                                   ineffective=self._ineffective(frame_hash))\n            img = frame_png_data_uri(grid) if self.use_image else None\n            self._llm_calls += 1\n            reply = self.chat_fn(SYSTEM_PROMPT, user, img)\n            actions = parse_actions(reply)\n        except Exception:\n            return None\n        # filtrar por legalidad e inefectividad conocida\n        legal = []\n        failed = self._failed.get(frame_hash, set())\n        for a in actions:\n            if available_actions and a[\"id\"] not in available_actions and a[\"id\"] != 6:\n                continue\n            if self._fail_key(a) in failed:\n                continue\n            legal.append(a)\n        if not legal:\n            return None\n        for a in legal[1:self.plan_max]:\n            self._plan.append(a)\n        return legal[0]\n\n    def _emit(self, grid: np.ndarray, action: dict[str, Any]) -> tuple[int, int, int]:\n        self._prev_grid = grid.copy()\n        self._prev_hash = _hash(grid)\n        self._prev_action = action\n        aid = action[\"id\"]\n        return aid, int(action.get(\"x\", -1)), int(action.get(\"y\", -1))\n",
 "src/arc3/llm_prompt.py": "\"\"\"Construcci\u00f3n de prompt e interpretaci\u00f3n de respuesta para el agente VLM.\n\nDiferenciador propio (ninguno del top lo hace): adem\u00e1s de la imagen del frame, se\ninyecta en el prompt una descripci\u00f3n TEXTUAL de la estructura del frame calculada con\nnuestras features objetuales (`arc3.features`): objetos (color, tama\u00f1o, bbox, \"bot\u00f3n-idad\"),\nfondo, y el efecto num\u00e9rico de la \u00faltima acci\u00f3n (p\u00edxeles cambiados, vector de movimiento).\nEl VLM razona sobre datos duros en vez de solo p\u00edxeles, que es donde alucina.\n\nTodo aqu\u00ed es puro (numpy + str): testeable sin GPU.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport base64\nimport io\nimport json\nfrom typing import Any, Optional\n\nimport numpy as np\n\nfrom .features import connected_components, transition_features\n\n# Paleta ARC de 16 colores (RGB) \u2014 la misma del visor oficial.\nARC_PALETTE = np.array([\n    (0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64), (255, 220, 0),\n    (170, 170, 170), (240, 18, 190), (255, 133, 27), (127, 219, 255), (135, 12, 37),\n    (100, 70, 30), (140, 100, 60), (90, 90, 90), (30, 30, 90), (200, 200, 255),\n    (255, 255, 255),\n], dtype=np.uint8)\n\n# Nombres sem\u00e1nticos de acciones para el LLM (mapeo id -> nombre y viceversa).\nACTION_NAMES = {1: \"up\", 2: \"down\", 3: \"left\", 4: \"right\", 5: \"action5\",\n                6: \"click\", 7: \"action7\"}\nNAME_TO_ID = {v: k for k, v in ACTION_NAMES.items()}\nNAME_TO_ID.update({\"a1\": 1, \"a2\": 2, \"a3\": 3, \"a4\": 4, \"a5\": 5, \"a6\": 6, \"a7\": 7,\n                   \"reset\": 0})\n\n\ndef render_frame_png(grid: np.ndarray, scale: int = 8) -> bytes:\n    \"\"\"Grid 64x64 -> PNG RGB escalado (para el canal visual del VLM).\"\"\"\n    from PIL import Image\n\n    rgb = ARC_PALETTE[np.clip(grid, 0, 15)]\n    img = Image.fromarray(rgb, \"RGB\").resize(\n        (grid.shape[1] * scale, grid.shape[0] * scale), Image.NEAREST)\n    buf = io.BytesIO()\n    img.save(buf, format=\"PNG\")\n    return buf.getvalue()\n\n\ndef frame_png_data_uri(grid: np.ndarray, scale: int = 8) -> str:\n    return \"data:image/png;base64,\" + base64.b64encode(render_frame_png(grid, scale)).decode()\n\n\ndef describe_objects(grid: np.ndarray, max_objects: int = 12) -> str:\n    \"\"\"Descripci\u00f3n textual compacta de los objetos del frame (nuestro diferenciador).\"\"\"\n    counts = np.bincount(grid.ravel(), minlength=16)\n    background = int(counts.argmax())\n    objs = connected_components(grid, background)\n    total = grid.size\n    lines = [f\"background_color={background}, distinct_colors={int((counts > 0).sum())}, \"\n             f\"objects={len(objs)}\"]\n    for i, o in enumerate(objs[:max_objects]):\n        y0, x0, y1, x1 = o[\"bbox\"]\n        cy, cx = o[\"centroid\"]\n        rarity = 1.0 - counts[o[\"color\"]] / total\n        area = (y1 - y0 + 1) * (x1 - x0 + 1)\n        buttonness = (0.5 * rarity + 0.5 * (o[\"size\"] / area)) * (1.0 if o[\"size\"] <= 64 else 0.3)\n        lines.append(\n            f\"  obj{i}: color={o['color']} size={o['size']} \"\n            f\"bbox=(x{x0}-{x1},y{y0}-{y1}) center=(x{int(cx)},y{int(cy)}) \"\n            f\"button_score={buttonness:.2f}\")\n    return \"\\n\".join(lines)\n\n\ndef describe_last_transition(prev: Optional[np.ndarray], cur: np.ndarray) -> str:\n    if prev is None:\n        return \"last_action_effect: (none, first frame)\"\n    tf = transition_features(prev, cur)\n    mv = \"\"\n    if tf[\"move_score\"] > 0.6 and (tf[\"move_dy\"] or tf[\"move_dx\"]):\n        mv = f\", object_moved=(dy{tf['move_dy']},dx{tf['move_dx']})\"\n    return (f\"last_action_effect: pixels_changed={tf['n_changed']}, \"\n            f\"colors_gained={tf['colors_gained']}, colors_lost={tf['colors_lost']}{mv}\")\n\n\nSYSTEM_PROMPT = (\n    \"You are an agent playing an interactive puzzle game on a 64x64 colored grid \"\n    \"(colors 0-15, coordinates x=0..63 left-to-right, y=0..63 top-to-bottom). \"\n    \"You explore to discover the rules, then act to complete levels. \"\n    \"Trust the numeric STRUCTURE and TRANSITION data over the image when they disagree. \"\n    \"Respond ONLY with a JSON object: {\\\"reasoning\\\": \\\"...\\\", \\\"actions\\\": [ ... ]} where \"\n    \"each action is either {\\\"name\\\": \\\"up|down|left|right|action5|action7\\\"} or \"\n    \"{\\\"name\\\": \\\"click\\\", \\\"x\\\": <0-63>, \\\"y\\\": <0-63>}. Plan 1-3 actions. Prefer actions \"\n    \"not marked ineffective. To find interactive elements, click objects with high button_score.\"\n)\n\n\ndef build_user_text(\n    grid: np.ndarray,\n    prev_grid: Optional[np.ndarray],\n    available_actions: list[int],\n    levels_completed: int,\n    ineffective: Optional[list[str]] = None,\n    memory: Optional[str] = None,\n) -> str:\n    \"\"\"Texto del turno: acciones legales + estructura de objetos + efecto de la \u00faltima acci\u00f3n.\"\"\"\n    legal = [ACTION_NAMES[a] for a in available_actions if a in ACTION_NAMES] or \\\n        list(ACTION_NAMES.values())\n    parts = [\n        f\"levels_completed={levels_completed}\",\n        f\"legal_actions={legal}\",\n        \"FRAME STRUCTURE:\",\n        describe_objects(grid),\n        describe_last_transition(prev_grid, grid),\n    ]\n    if ineffective:\n        parts.append(f\"ineffective_in_this_state={ineffective[:20]}\")\n    if memory:\n        parts.append(f\"MEMORY:\\n{memory}\")\n    parts.append(\"Return your JSON now.\")\n    return \"\\n\".join(parts)\n\n\ndef parse_actions(text: str) -> list[dict[str, Any]]:\n    \"\"\"Extrae la lista de acciones del texto del LLM de forma robusta.\n\n    Escanea todos los objetos JSON del texto y prioriza el que tenga clave 'actions'.\n    Devuelve lista de dicts normalizados: {'id': int, 'x': int?, 'y': int?}.\n    \"\"\"\n    obj = _extract_json_with_actions(text)\n    raw_actions = []\n    if obj and isinstance(obj.get(\"actions\"), list):\n        raw_actions = obj[\"actions\"]\n    elif obj and \"name\" in obj:  # el LLM devolvi\u00f3 una sola acci\u00f3n suelta\n        raw_actions = [obj]\n    out: list[dict[str, Any]] = []\n    for a in raw_actions:\n        if not isinstance(a, dict):\n            continue\n        name = str(a.get(\"name\", a.get(\"action\", \"\"))).strip().lower()\n        aid = NAME_TO_ID.get(name)\n        if aid is None:\n            continue\n        entry: dict[str, Any] = {\"id\": aid}\n        if aid == 6:\n            try:\n                entry[\"x\"] = max(0, min(63, int(a.get(\"x\", 32))))\n                entry[\"y\"] = max(0, min(63, int(a.get(\"y\", 32))))\n            except (TypeError, ValueError):\n                entry[\"x\"] = entry[\"y\"] = 32\n        out.append(entry)\n    return out\n\n\ndef _extract_json_with_actions(text: str) -> Optional[dict[str, Any]]:\n    dec = json.JSONDecoder()\n    best: Optional[dict[str, Any]] = None\n    i = 0\n    n = len(text)\n    while i < n:\n        c = text[i]\n        if c != \"{\":\n            i += 1\n            continue\n        try:\n            obj, end = dec.raw_decode(text, i)\n        except json.JSONDecodeError:\n            i += 1\n            continue\n        if isinstance(obj, dict):\n            if \"actions\" in obj:\n                return obj\n            if best is None and \"name\" in obj:\n                best = obj\n        i = end\n    return best\n",
 "src/arc3/probe.py": "\"\"\"Pol\u00edtica de sondeo: juega cada environment y produce el dataset de features.\n\nEstrategia por juego:\n  1. RESET y features del frame inicial.\n  2. Round-robin sobre las acciones simples disponibles (ACTION1..5, 7) para\n     perfilar qu\u00e9 hace cada una (\u00bfcambia el frame?, \u00bfmueve un objeto?, \u00bfsube nivel?).\n  3. Sondeo de ACTION6 (click x,y) sobre una malla gruesa de puntos, para mapear\n     regiones interactivas.\n  4. Si el juego llega a GAME_OVER se hace RESET y se contin\u00faa hasta agotar budget.\n\nCada paso emite una fila con features de transici\u00f3n + features del frame resultante.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport random\nimport time\nfrom typing import Any, Optional\n\nfrom arcengine import FrameDataRaw, GameAction, GameState\n\nfrom .env import LocalGame\nfrom .features import frame_to_grid, grid_features, transition_features\n\nSIMPLE_ACTIONS = [\n    GameAction.ACTION1,\n    GameAction.ACTION2,\n    GameAction.ACTION3,\n    GameAction.ACTION4,\n    GameAction.ACTION5,\n    GameAction.ACTION7,\n]\n\n\ndef _click_grid(n: int = 8) -> list[tuple[int, int]]:\n    \"\"\"Malla n x n de puntos (x, y) centrados en tiles de 64/n.\"\"\"\n    step = 64 // n\n    half = step // 2\n    return [(x * step + half, y * step + half) for y in range(n) for x in range(n)]\n\n\ndef probe_game(\n    game: LocalGame,\n    budget: int = 300,\n    click_grid_n: int = 8,\n    seed: int = 0,\n    time_limit_s: Optional[float] = None,\n) -> list[dict[str, Any]]:\n    \"\"\"Sondea un juego y devuelve filas de features (una por acci\u00f3n ejecutada).\"\"\"\n    rng = random.Random(seed)\n    rows: list[dict[str, Any]] = []\n    t0 = time.time()\n\n    frame = game.reset()\n    if frame is None:\n        return rows\n    prev_grid = frame_to_grid(frame.frame)\n    prev_levels = frame.levels_completed\n\n    clicks = _click_grid(click_grid_n)\n    rng.shuffle(clicks)\n    click_i = 0\n    step_i = 0\n\n    while step_i < budget:\n        if time_limit_s is not None and time.time() - t0 > time_limit_s:\n            break\n        avail = frame.available_actions or []\n        simple = [a for a in SIMPLE_ACTIONS if not avail or a.value in avail]\n        use_click = (GameAction.ACTION6.value in avail or not avail) and (\n            not simple or step_i % 3 == 2\n        )\n\n        x = y = None\n        if use_click and click_i < len(clicks):\n            action = GameAction.ACTION6\n            x, y = clicks[click_i]\n            click_i += 1\n        elif simple:\n            action = simple[step_i % len(simple)]\n        elif GameAction.ACTION6.value in avail:\n            action = GameAction.ACTION6\n            x, y = rng.randrange(64), rng.randrange(64)\n        else:\n            break\n\n        nxt = game.step(action, x=x, y=y)\n        step_i += 1\n        if nxt is None:\n            continue\n\n        next_grid = frame_to_grid(nxt.frame)\n        row: dict[str, Any] = {\n            \"game_id\": game.info.game_id,\n            \"step\": step_i,\n            \"action_id\": action.value,\n            \"click_x\": -1 if x is None else x,\n            \"click_y\": -1 if y is None else y,\n            \"state\": nxt.state.value,\n            \"levels_completed\": nxt.levels_completed,\n            \"win_levels\": nxt.win_levels,\n            \"level_up\": nxt.levels_completed > prev_levels,\n            \"game_over\": nxt.state == GameState.GAME_OVER,\n            \"win\": nxt.state == GameState.WIN,\n        }\n        row.update(transition_features(prev_grid, next_grid))\n        row.update({f\"nf_{k}\": v for k, v in grid_features(next_grid).items()})\n        rows.append(row)\n\n        prev_levels = nxt.levels_completed\n        prev_grid = next_grid\n        frame = nxt\n\n        if nxt.state in (GameState.GAME_OVER, GameState.WIN):\n            frame = game.reset() or frame\n            prev_grid = frame_to_grid(frame.frame)\n            prev_levels = frame.levels_completed\n\n    return rows\n",
 "src/arc3/runner.py": "\"\"\"Runner paralelo de juegos ARC-AGI-3 sobre un Arcade (offline o gateway).\n\nEn el rerun real cada acci\u00f3n es un request HTTP al gateway (latencia-bound): jugar\nN juegos en paralelo multiplica el throughput de acciones (el milestone winner usaba\nconcurrencia 28). Offline es CPU-bound: pocos workers bastan.\n\nUso (notebook de submission y eval local):\n    arcade = Arcade(operation_mode=..., ...)\n    results = run_games(arcade, game_ids, total_budget_s=..., workers=12)\n\"\"\"\n\nfrom __future__ import annotations\n\nimport threading\nimport time\nfrom typing import Any, Callable, Optional\n\nfrom arcengine import GameAction\n\nfrom .agent import GraphExplorer\nfrom .features import frame_to_grid\n\n# F\u00e1brica de agente inyectable: el notebook LLM la reemplaza por una que crea LLMAgent.\n# Firma: (game_id: str, max_actions: int) -> agente con .choose(...) y .done.\n_AGENT_FACTORY: Optional[Callable[[str, int], Any]] = None\n\n\ndef play_game(\n    env: Any,\n    game_id: str,\n    time_budget_s: float,\n    max_actions: int = 15000,\n    stop_event: Optional[threading.Event] = None,\n) -> dict[str, Any]:\n    \"\"\"Juega un env (EnvironmentWrapper de arc_agi) hasta agotar budget.\n\n    Usa _AGENT_FACTORY si est\u00e1 definida (LLMAgent), si no GraphExplorer.\n    \"\"\"\n    agent = (_AGENT_FACTORY or (lambda gid, ma: GraphExplorer(gid, max_actions=ma)))(\n        game_id, max_actions)\n    t0 = time.time()\n    try:\n        frame = env.observation_space or env.reset()\n    except Exception:\n        frame = None\n    best = 0\n    win = False\n    while (\n        frame is not None\n        and not agent.done\n        and time.time() - t0 < time_budget_s\n        and not (stop_event and stop_event.is_set())\n    ):\n        try:\n            grid = frame_to_grid(frame.frame)\n            aid, x, y = agent.choose(\n                grid, frame.state.value, frame.levels_completed,\n                list(frame.available_actions or []),\n            )\n            action = GameAction.from_id(aid)\n            data: dict[str, Any] = {\"game_id\": game_id}\n            if aid == 6:\n                data.update(x=x, y=y)\n            frame = env.reset() if aid == 0 else env.step(action, data=data)\n        except Exception:\n            try:\n                frame = env.reset()\n            except Exception:\n                break\n        if frame is not None:\n            best = max(best, frame.levels_completed)\n            if frame.state.value == \"WIN\":\n                win = True\n                break\n    return {\n        \"game_id\": game_id,\n        \"levels\": best,\n        \"win\": win,\n        \"actions\": getattr(agent, \"actions_taken\", 0),\n        \"seconds\": round(time.time() - t0, 1),\n        \"nodes\": len(getattr(agent, \"_nodes\", ()) or ()),   # GraphExplorer; LLMAgent no tiene\n        \"llm_calls\": getattr(agent, \"_llm_calls\", 0),\n        \"llm_fails\": getattr(agent, \"_llm_fails\", 0),\n    }\n\n\ndef run_games(\n    arcade: Any,\n    game_ids: list[str],\n    total_budget_s: float,\n    workers: int = 8,\n    max_actions: int = 15000,\n    max_game_s: float = 1800.0,\n    min_game_s: float = 60.0,\n    card_id: Optional[str] = None,\n    verbose: bool = True,\n) -> list[dict[str, Any]]:\n    \"\"\"Juega todos los game_ids con un pool de threads y presupuesto global compartido.\"\"\"\n    t_start = time.time()\n    results: list[dict[str, Any]] = []\n    queue = list(game_ids)\n    lock = threading.Lock()\n    stop_event = threading.Event()\n\n    def remaining() -> float:\n        return total_budget_s - (time.time() - t_start)\n\n    def worker() -> None:\n        while not stop_event.is_set():\n            with lock:\n                if not queue:\n                    return\n                games_left = len(queue)\n                game_id = queue.pop(0)\n            rem = remaining()\n            if rem < min_game_s:\n                stop_event.set()\n                return\n            # presupuesto por juego: reparte el tiempo restante entre los juegos que\n            # quedan, multiplicado por los workers (corren en paralelo)\n            budget = max(min_game_s, min(max_game_s, rem * workers / max(games_left, 1)))\n            budget = min(budget, rem)\n            try:\n                with lock:\n                    env = arcade.make(game_id, scorecard_id=card_id)\n                if env is None:\n                    raise RuntimeError(\"make() devolvi\u00f3 None\")\n                r = play_game(env, game_id, budget, max_actions, stop_event)\n            except Exception as e:\n                r = {\"game_id\": game_id, \"levels\": 0, \"win\": False, \"actions\": 0,\n                     \"seconds\": 0.0, \"nodes\": 0, \"error\": str(e)[:200]}\n            with lock:\n                results.append(r)\n                if verbose:\n                    print(f\"[{len(results)}/{len(game_ids)}] {r['game_id']}: \"\n                          f\"{r['levels']} niveles, {r['actions']} acciones, \"\n                          f\"{r['seconds']}s{' WIN' if r.get('win') else ''}\"\n                          f\"{' ERROR ' + r['error'] if r.get('error') else ''}\",\n                          flush=True)\n\n    threads = [threading.Thread(target=worker, daemon=True) for _ in range(workers)]\n    for t in threads:\n        t.start()\n    for t in threads:\n        t.join(timeout=max(0.0, total_budget_s - (time.time() - t_start)) + max_game_s)\n    if verbose:\n        total = sum(r[\"levels\"] for r in results)\n        print(f\"TOTAL: {total} niveles en {len(results)} juegos \"\n              f\"({time.time() - t_start:.0f}s)\", flush=True)\n    return results\n"
}
for path, code in SOURCES.items():
    open('/kaggle/working/'+path,'w',encoding='utf-8').write(code)
print('src/arc3 reconstruido')


In [ ]:
# Instalar vLLM del wheelhouse público y arrancar el server OpenAI-compatible.
VLLM_HOST, VLLM_PORT = "127.0.0.1", 1234
VLLM_BASE = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
SERVED = "arc-qwen"
SERVER_LOG = Path("/kaggle/working/vllm.log")

def find_wheelhouse():
    for dp, dns, fns in os.walk("/kaggle/input"):
        if any(f.startswith("vllm-") and f.endswith(".whl") for f in fns):
            return Path(dp)
    return None

wh = find_wheelhouse()
if wh is not None and MODEL_DIR is not None:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-index",
        f"--find-links={wh}","vllm"], stdout=subprocess.DEVNULL)
    # El crash previo fue el autotuner de flashinfer al capturar el grafo. --enforce-eager
    # salta torch.compile + cudagraph (donde se dispara el autotune); FLASH_ATTN evita la
    # ruta flashinfer de atención; desactivamos el sampler flashinfer. max-model-len 16k
    # basta para nuestros prompts cortos y baja presión de KV.
    # El crash es el kernel GEMM FP8 de flashinfer (FlashInferFP8ScaledMMLinearKernel):
    # su autotuner JIT muere. Forzamos el kernel Marlin FP8 (Triton, sin flashinfer) con
    # VLLM_TEST_FORCE_FP8_MARLIN=1. Sumado a FLASH_ATTN + enforce-eager, no queda ninguna
    # ruta que dependa de flashinfer.
    serve_env = os.environ.copy()
    serve_env.update({
        "VLLM_TEST_FORCE_FP8_MARLIN": "1",
        "VLLM_ATTENTION_BACKEND": "FLASH_ATTN",
        "VLLM_USE_FLASHINFER_SAMPLER": "0",
        "VLLM_NO_USAGE_STATS": "1",
    })
    cmd = [sys.executable,"-m","vllm.entrypoints.openai.api_server",
           "--model", str(MODEL_DIR), "--served-model-name", SERVED,
           "--host", VLLM_HOST, "--port", str(VLLM_PORT),
           "--tensor-parallel-size","1","--enforce-eager","--enable-prefix-caching",
           "--max-model-len","16384","--gpu-memory-utilization","0.90"]
    print("arrancando vLLM:", " ".join(cmd), flush=True)
    logf = SERVER_LOG.open("w")
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=serve_env)
    # esperar readiness
    ready = False
    dl = time.monotonic() + 900
    while time.monotonic() < dl:
        try:
            with urlopen(f"{VLLM_BASE}/models", timeout=5) as r:
                if r.status == 200:
                    ready = True; break
        except Exception:
            pass
        if proc.poll() is not None:
            print("vLLM murió:\n", SERVER_LOG.read_text()[-2000:]); break
        time.sleep(5)
    print("vLLM ready:", ready, flush=True)
else:
    ready = False
    print("sin wheelhouse/modelo -> el agente correrá SOLO con fallback GraphExplorer")


In [ ]:
# chat_fn: cliente OpenAI-compatible contra el vLLM local (texto; imagen opcional).
import urllib.request

def chat_fn(system, user_text, image_data_uri):
    content = [{"type":"text","text":user_text}]
    if image_data_uri:
        content.append({"type":"image_url","image_url":{"url":image_data_uri}})
    body = {"model": SERVED, "messages":[
                {"role":"system","content":system},
                {"role":"user","content": content if image_data_uri else user_text}],
            "max_tokens":800,"temperature":0.3,
            # Qwen3 es modelo de razonamiento: sin esto emite <think> largo y el JSON
            # de acciones se trunca. enable_thinking=False -> responde directo.
            "chat_template_kwargs":{"enable_thinking":False}}
    req = urllib.request.Request(f"{VLLM_BASE}/chat/completions",
            data=json.dumps(body).encode(), headers={"Content-Type":"application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.loads(r.read())
    return out["choices"][0]["message"]["content"]

# SMOKE TEST: una llamada real, imprime respuesta cruda + acciones parseadas (barato,
# evita gastar la corrida entera a ciegas si el formato del modelo no encaja).
if ready:
    import numpy as np
    sys.path.insert(0, "/kaggle/working/src")
    from arc3.llm_prompt import SYSTEM_PROMPT, build_user_text, parse_actions
    g = np.zeros((64,64), dtype=np.int8); g[10:14,20:24]=2; g[40,40]=5
    smoke_user = build_user_text(g, None, [1,2,3,4,6], 0)
    try:
        raw = chat_fn(SYSTEM_PROMPT, smoke_user, None)
        print("SMOKE raw (first 500):", repr(raw[:500]), flush=True)
        print("SMOKE parsed actions:", parse_actions(raw), flush=True)
    except Exception as e:
        print("SMOKE error:", e, flush=True)


In [ ]:
sys.path.insert(0, "/kaggle/working/src")
from arc_agi.base import Arcade, OperationMode
from arc3.runner import run_games, play_game
from arc3.llm_agent import LLMAgent
import arc3.runner as runner_mod

# inyectar LLMAgent (con nuestro chat_fn) como política del runner, GraphExplorer de fallback
USE_IMAGE = os.environ.get("ARC_USE_IMAGE","0") == "1"
def make_agent(game_id, max_actions):
    if ready:
        return LLMAgent(game_id, chat_fn, max_actions=max_actions, use_image=USE_IMAGE)
    from arc3.agent import GraphExplorer
    return GraphExplorer(game_id, max_actions=max_actions)
runner_mod._AGENT_FACTORY = make_agent  # play_game lo usa si existe

if TRUE_SUBMISSION:
    base = os.environ.get("ARC_BASE_URL","http://gateway:8001/")
    dl=time.monotonic()+600
    while time.monotonic()<dl:
        try:
            with urlopen(base+"api/games", timeout=10) as r:
                if r.status<500: break
        except Exception: pass
        time.sleep(5)
    arcade = Arcade(arc_api_key=os.environ.get("ARC_API_KEY","test-key-123"),
                    arc_base_url=base, operation_mode=OperationMode.COMPETITION, environments_dir="")
    BUDGET = 8*3600-900 - (time.time()-NOTEBOOK_START); WORKERS=8; MAXG=3600.0
else:
    arcade = Arcade(operation_mode=OperationMode.OFFLINE,
                    environments_dir=str(COMP_ROOT/"environment_files"))
    # vLLM agrupa (continuous batching) los requests concurrentes: subir workers eleva el
    # throughput del LLM en la MISMA GPU. Clave para que el LLM haga suficientes acciones.
    BUDGET = OFFLINE_SOFT_MIN*60; WORKERS=8; MAXG=600.0

game_ids=[e.game_id for e in arcade.available_environments]
try: card_id = arcade.open_scorecard(tags=["agent","llm"])
except Exception as e: print("scorecard:",e); card_id=None

import pandas as pd
pd.DataFrame([["1_0","1",True,1]],columns=["row_id","game_id","end_of_game","score"]).to_parquet("/kaggle/working/submission.parquet",index=False)

results = run_games(arcade, game_ids, total_budget_s=BUDGET, workers=WORKERS,
                    max_actions=15000, max_game_s=MAXG, card_id=card_id)
pd.DataFrame(results).to_csv("/kaggle/working/results.csv", index=False)
print("TOTAL niveles:", sum(r["levels"] for r in results))
